![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4D: Lead Research and Writing Agent

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock research-and-writing agent using approved public snippets</td></tr>
<tr><td align="left">Optional part</td><td>Real model drafting if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A short evidence-grounded lead-research summary and outreach draft</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04d-overview)
2. [Setup and Background](#m04d-setup)
3. [Core Concepts](#m04d-evidence)
4. [Guided Implementation](#m04d-agent)
5. [Testing and Analysis](#m04d-testing)
6. [Student Tasks](#m04d-student-tasks)
7. [Submission and Reflection](#m04d-submission)

---

<a id="m04d-overview"></a>

### 1. Overview and Learning Goals

M04D combines the main ideas of this module into a small end-to-end agentic workflow. M04A gave you prompt-model-parser chains, M04B gave you controlled tools, and M04C gave you safe local storage and state-changing actions. M04D assembles these ideas into a **lead research and writing agent** — a multi-step pipeline where the output of one stage becomes the input of the next.

In this notebook, "lead research" does not mean scraping the web or contacting real people. It means analysing a small approved set of public snippets and producing a structured research summary plus a draft outreach message. The agent will not send email; it only drafts text for a human to review.

A useful analogy: the agent is a diligent junior research assistant. It may only use the folder of approved clippings on its desk, it must attach a citation to every claim it makes, it must say so plainly when the folder contains nothing relevant, and it hands its draft letter to a human rather than posting it.

```text
Research request
      |
      v
+---------------+     +------------------------+     +------------------+
| Safety check  | --> | Search approved public | --> | Build structured |
| (refuse or    |     | snippets (evidence)    |     | summary          |
|  continue)    |     +------------------------+     +------------------+
+---------------+                                            |
      |                                                      v
      | unsafe request                          +------------------------+
      v                                         | Draft outreach message |
   Refusal                                      | (requires_review=True) |
                                                +------------------------+
                                                             |
                                                             v
                                                       Human review
```

This design is intentionally conservative. Many real lead-generation systems connect to search engines, CRMs, email systems, calendars and analytics — higher-risk external actions that belong to later modules. In this teaching lab, the workflow stays local and controlled so you can focus on the architecture, on evidence use, and on safe drafting.

By the end of this session, you should be able to build a small research-and-writing pipeline, use only approved evidence, produce structured output, avoid unsupported claims, test boundary cases, and explain how this prepares for more advanced agent workflows in M05 and M08.

<a id="m04d-setup"></a>

### 2. Setup and Background

#### 2.1 What is a research-and-writing agent?

A research-and-writing agent is a workflow that collects relevant information, organises it, and drafts text for a human to review. It should not be treated as an autonomous salesperson or decision-maker: the final message must always be checked by a human before use, because a fluent draft can be persuasive even when its facts are thin.

In this notebook, the agent works with a small in-memory evidence store. It can:

```text
1. search approved public snippets,
2. select relevant evidence,
3. build a structured summary,
4. draft a short outreach message,
5. flag limitations and missing evidence.
```

It cannot:

```text
1. browse the live web,
2. contact real people,
3. send email,
4. access private databases,
5. invent unsupported claims,
6. use hidden instructor materials.
```

#### 2.2 Why this session belongs after M04C

M04C showed that state-changing actions require validation. M04D adds a complementary idea: writing agents require **evidence control**. A generated outreach draft can sound confident and professional even when nothing supports it, so the workflow must make its evidence — and the absence of evidence — visible at every step.

```text
Evidence found:   Evidence --> Summary --> Draft ------------> Human review
                              (cites its    (uses only the
                               sources)      cited evidence)

No evidence:      State the limitation clearly --------------> Human review
                  (never pad the gap with invented claims)
```

Both paths end at human review. The difference between a trustworthy writing agent and a risky one is what happens on the lower path: a trustworthy agent says "I found nothing", while a risky one fills the silence with plausible fiction.

In [ ]:
# Standard-library imports only: the mandatory research-and-writing pipeline
# must run with no installation, no API key and no internet access.
import json
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

print("M04D setup complete.")

<a id="m04d-evidence"></a>

### 3. Core Concepts

#### 3.1 Public Evidence Store

The evidence store below is a local teaching dataset. It contains short public-style snippets about fictional organisations and their AI-related interests. In a real system, evidence might come from public websites, public reports or approved CRM notes; in this notebook we use a controlled local dataset to avoid privacy, scraping and external-action risks — and to make every retrieval result fully explainable.

Each evidence item has:

```text
lead_id: identifier
organisation: organisation name
sector: broad sector
snippet: public-style evidence text
tags: searchable topic tags
source: source description or URL placeholder
```

The `source` field matters most: it is what turns a claim in a draft into something a human reviewer can check. Carrying the source alongside the text, from retrieval all the way into the final draft, is the habit this section builds.

In [ ]:
# Approved evidence store: five fictional, public-style leads.
# Kept deliberately small so every retrieval result can be traced by eye.
# Note that every item carries a "source" field - it travels with the snippet
# so the final draft can always point back to where a claim came from.
PUBLIC_EVIDENCE = [
    {
        "lead_id": "L001",
        "organisation": "Northbank Health Analytics",
        "sector": "health",
        "snippet": "Northbank Health Analytics publishes public reports on hospital demand forecasting and responsible AI governance.",
        "tags": ["health", "forecasting", "responsible_ai", "governance"],
        "source": "public report summary"
    },
    {
        "lead_id": "L002",
        "organisation": "Harbour Retail Group",
        "sector": "retail",
        "snippet": "Harbour Retail Group has discussed customer-service automation, product recommendation and privacy-preserving analytics in public innovation updates.",
        "tags": ["retail", "recommendation", "customer_service", "privacy"],
        "source": "public innovation update"
    },
    {
        "lead_id": "L003",
        "organisation": "GreenField Smart Farming",
        "sector": "agriculture",
        "snippet": "GreenField Smart Farming explores sensor-based crop monitoring, weather-aware decision support and AI-assisted irrigation planning.",
        "tags": ["agriculture", "smart_farming", "iot", "decision_support"],
        "source": "public project description"
    },
    {
        "lead_id": "L004",
        "organisation": "MetroCyber Training Institute",
        "sector": "education",
        "snippet": "MetroCyber Training Institute offers public short courses on cybersecurity awareness, AI safety and digital-skills training.",
        "tags": ["education", "cybersecurity", "ai_safety", "training"],
        "source": "public course catalogue"
    },
    {
        "lead_id": "L005",
        "organisation": "Civic Transport Lab",
        "sector": "transport",
        "snippet": "Civic Transport Lab publishes open material on traffic prediction, route optimisation and data-driven transport planning.",
        "tags": ["transport", "prediction", "optimisation", "planning"],
        "source": "public research page"
    },
]

len(PUBLIC_EVIDENCE), PUBLIC_EVIDENCE[0]

The evidence store is intentionally small — five items you can read in a minute. That is a feature, not a limitation: with a store this size you can inspect exactly why the agent matched or missed a lead, which is impossible with a live search index. A real agent should preserve the same habit at scale: show which evidence was used, carry sources with claims, and avoid statements that no stored item supports.

In [ ]:
def normalise_text(text: str) -> str:
    # Lower-case and collapse whitespace so matching is not defeated by
    # capitalisation or stray spaces.
    return re.sub(r"\s+", " ", text.lower()).strip()


def search_evidence(query: str, evidence_store: List[Dict[str, Any]], top_k: int = 3) -> Dict[str, Any]:
    """Search the approved evidence store using simple keyword overlap.

    Scoring: count how many words the query and an item share. Design
    decisions worth noticing:
    - items with score 0 are excluded entirely, so an off-topic query returns
      an empty list rather than the "least bad" match;
    - top_k defaults to 3 to keep summaries focused - raising it adds weaker
      matches, lowering it may drop useful context;
    - every input is checked first, in the usual ok/error/result envelope.
    """

    if not isinstance(query, str) or not query.strip():
        return {"ok": False, "error": "query must be a non-empty string.", "result": None}

    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    query_terms = set(re.findall(r"[a-zA-Z_]+", normalise_text(query)))
    scored = []

    for item in evidence_store:
        # Search across all text-bearing fields, so a query can match the
        # organisation name, the sector, the snippet or a tag.
        item_text = " ".join([
            item.get("organisation", ""),
            item.get("sector", ""),
            item.get("snippet", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(re.findall(r"[a-zA-Z_]+", normalise_text(item_text)))
        score = len(query_terms.intersection(item_terms))
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = [item for _, item in scored[:top_k]]

    return {"ok": True, "error": None, "result": results}


search_evidence("AI safety training education", PUBLIC_EVIDENCE)

This search function is simple on purpose. It is not a vector database and not a live search engine; it counts shared words. Its role is to demonstrate the evidence-selection step in a form you can verify by hand: for the query above, you can check yourself that the MetroCyber item shares the words "ai", "safety", "training" and "education". Notice also what happens with an unrelated query — no item scores above zero, so the result is an empty list, and the rest of the pipeline must handle that honestly. Later RAG sessions in M05 replace this word counting with embeddings and vector search, but the pipeline position and the empty-result obligation stay the same.

<a id="m04d-agent"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Local Research and Writing Agent

The local agent runs a fixed three-stage pipeline, guarded by a safety check:

```text
request --> safety check --> 1. retrieve relevant approved evidence
                                        |
                                        v
                             2. build a structured summary
                                (matched leads + themes + limitations)
                                        |
                                        v
                             3. draft an outreach message
                                (grounded in stage-2 evidence only,
                                 marked requires_review=True)
```

Each stage consumes only the previous stage's structured output — the drafting function never looks at the evidence store directly, only at what the summary passed to it. That hand-off discipline is what keeps the final draft grounded: a claim can only appear in the draft if it survived retrieval and summarisation first.

The agent refuses unsafe requests such as contacting real people, sending email, scraping private websites or accessing credentials. You will build the safety check first, then the two writing stages, then wire everything together.

In [ ]:
def check_request_safety(request: str) -> Dict[str, Any]:
    """Classify a request as safe (research and drafting) or unsafe.

    Runs BEFORE any retrieval or drafting, mirroring M04B/M04C: an unsafe
    request should not be partially served. The result separates "the check
    ran" (ok) from "the request is allowed" (result.safe) - a request can be
    validly checked and still be refused.
    """
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()
    # Keyword lists are a teaching simplification; real systems layer
    # permissions, policy models and human review on top.
    unsafe_keywords = [
        "send email", "send message", "contact them", "scrape", "private database",
        "password", "api key", "credential", "hidden solution", "student record",
        "personal phone", "personal email"
    ]

    if any(keyword in lower for keyword in unsafe_keywords):
        return {
            "ok": True,
            "error": None,
            "result": {
                "safe": False,
                "reason": "The request asks for external contact, private data, credentials, scraping, or hidden material."
            }
        }

    return {"ok": True, "error": None, "result": {"safe": True, "reason": "Allowed local research-and-drafting request."}}


print(check_request_safety("Draft an outreach message for AI safety training."))
print(check_request_safety("Send email to all leads."))

In [ ]:
def build_research_summary(query: str, evidence_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Build a structured research summary from retrieved evidence.

    Two deliberate design choices:
    - the empty-evidence case is handled FIRST and produces an honest
      "nothing found" summary instead of padding the gap;
    - every matched lead keeps its source field, so claims remain checkable
      all the way to the draft.
    """

    if not evidence_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "query": query,
                "matched_leads": [],
                "summary": "No sufficiently relevant approved evidence was found.",
                "limitations": ["The approved evidence store may not contain this topic."],
            }
        }

    matched = []
    for item in evidence_items:
        matched.append({
            "lead_id": item["lead_id"],
            "organisation": item["organisation"],
            "sector": item["sector"],
            "evidence": item["snippet"],
            "source": item["source"],
        })

    # Collect tags across matches to surface the common themes; sorting keeps
    # the output deterministic and therefore testable.
    themes = sorted(set(tag for item in evidence_items for tag in item.get("tags", [])))

    summary = (
        f"Found {len(evidence_items)} relevant lead(s) for the query. "
        f"Common evidence themes include: {', '.join(themes[:8])}."
    )

    # Limitations are part of the OUTPUT, not an afterthought: whoever reads
    # this summary sees its boundaries alongside its findings.
    return {
        "ok": True,
        "error": None,
        "result": {
            "query": query,
            "matched_leads": matched,
            "summary": summary,
            "limitations": [
                "This summary uses only the approved local evidence store.",
                "It should not be treated as live web research.",
                "A human should verify all claims before use."
            ],
        }
    }


evidence = search_evidence("AI safety training education", PUBLIC_EVIDENCE)["result"]
summary = build_research_summary("AI safety training education", evidence)
summary

In [ ]:
def draft_outreach_message(summary_result: Dict[str, Any], audience: str = "potential collaborator") -> Dict[str, Any]:
    """Draft a short outreach message based only on structured summary evidence.

    Grounding rule: the draft may use only what arrived in summary_result.
    The function never reaches back into the evidence store, so nothing can
    appear in the message that did not pass retrieval and summarisation first.
    """

    if not isinstance(summary_result, dict) or "matched_leads" not in summary_result:
        return {"ok": False, "error": "summary_result is not valid.", "result": None}

    matched = summary_result["matched_leads"]

    # No evidence -> an honest non-draft. Writing a generic message anyway
    # would be exactly the unsupported-claim behaviour this lab teaches against.
    if not matched:
        return {
            "ok": True,
            "error": None,
            "result": {
                "draft": (
                    "I could not find enough approved evidence to draft a specific outreach message. "
                    "Please refine the topic or provide approved public context."
                ),
                "requires_review": True,
                "evidence_used": [],
            }
        }

    # The draft addresses the single best-matched lead and quotes its evidence
    # directly, so every claim in the message is traceable to one snippet.
    first = matched[0]
    draft = (
        f"Dear {first['organisation']} team,\n\n"
        f"I am reaching out regarding possible collaboration around {summary_result['query']}. "
        f"I noticed public information indicating your interest in {first['evidence']} "
        f"This appears aligned with practical work in agentic AI, responsible AI and applied data-driven systems.\n\n"
        f"If relevant, I would be pleased to discuss whether there is scope for a short exploratory conversation.\n\n"
        f"Kind regards,"
    )

    # requires_review is hard-coded True: no path through this function
    # produces a message that claims to be ready to send.
    return {
        "ok": True,
        "error": None,
        "result": {
            "draft": draft,
            "requires_review": True,
            "evidence_used": [first],
        }
    }


draft = draft_outreach_message(summary["result"])
print(draft["result"]["draft"])

The draft uses only the selected evidence: it names the organisation, refers to its published interests, and stops there. It does not invent personal contact details, does not claim private access, and does not send anything. Every result also carries `requires_review: True` and an `evidence_used` list — the two fields a human reviewer needs in order to check the message before it goes anywhere. Read the printed draft and confirm you can trace each factual statement back to the snippet it came from; that traceability is the entire point.

In [ ]:
class LeadResearchWritingAgent:
    """Controlled local research-and-writing agent using approved evidence only.

    The invoke method is a fixed pipeline with fail-fast hand-offs: safety
    check, then retrieval, then summary, then draft. Each stage receives only
    the previous stage's structured output, and any stage that fails stops the
    pipeline with its own error - the same chain discipline as M04A.
    """

    def __init__(self, evidence_store: List[Dict[str, Any]]):
        self.evidence_store = evidence_store

    def invoke(self, request: str, top_k: int = 3) -> Dict[str, Any]:
        # Stage 0: safety gate. Unsafe requests never reach retrieval.
        safety = check_request_safety(request)
        if not safety["ok"]:
            return safety

        if not safety["result"]["safe"]:
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "answer": safety["result"]["reason"],
                }
            }

        # Stage 1: retrieval from the approved store only.
        retrieved = search_evidence(request, self.evidence_store, top_k=top_k)
        if not retrieved["ok"]:
            return retrieved

        # Stage 2: structured summary (handles the empty-evidence case itself).
        summary = build_research_summary(request, retrieved["result"])
        if not summary["ok"]:
            return summary

        # Stage 3: grounded draft from the summary alone.
        draft = draft_outreach_message(summary["result"])
        if not draft["ok"]:
            return draft

        # Both intermediate products are returned, not just the final text:
        # the summary is the audit trail for the draft.
        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "research_and_draft",
                "summary": summary["result"],
                "draft": draft["result"],
            }
        }


agent = LeadResearchWritingAgent(PUBLIC_EVIDENCE)
agent_result = agent.invoke("AI safety training education")
agent_result

In [ ]:
def display_research_agent_result(agent_result: Dict[str, Any]) -> None:
    # Print in pipeline order - summary, evidence, limitations, draft - so the
    # reader can check each claim in the draft against the evidence above it.
    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))

    if result.get("action") == "refuse":
        print("Answer:", result.get("answer"))
        return

    summary = result["summary"]
    print("\nSummary:")
    print(summary["summary"])

    print("\nMatched leads:")
    for lead in summary["matched_leads"]:
        print(f"- {lead['lead_id']} | {lead['organisation']} | {lead['sector']}")
        print(f"  Evidence: {lead['evidence']}")

    print("\nLimitations:")
    for limitation in summary["limitations"]:
        print("-", limitation)

    print("\nDraft:")
    print(result["draft"]["draft"])


display_research_agent_result(agent_result)

The output is intentionally inspectable, and the display order is not accidental: evidence first, limitations next, draft last, so you naturally read the support before the claim. A student — or a reviewer — should be able to see the evidence, the summary, the limitations and the draft in one place. If any sentence in the draft is unsupported by the listed evidence, that mismatch should be visible immediately. This inspect-before-trust habit is the writing-agent equivalent of the debug view from M04A.

<a id="m04d-real"></a>

#### 4.2 Optional Real Model Drafting

This optional section shows how a real model could be used to rewrite the local draft more fluently. It is not required for the core task; use it only if you have a valid API key, internet access and the relevant packages installed. If not, record the section as skipped.

The key design point survives the upgrade: the real model is given the evidence and the draft, and instructed to rewrite *only* from that evidence — it polishes wording, it does not add facts. Even so, a rewritten draft still ends at human review, because instructions reduce unsupported claims but do not guarantee their absence.

In [ ]:
# Optional installation cell.
# Commented out on purpose so "Run all" never installs packages as a side
# effect. Uncomment only when package installation is allowed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
def optional_real_rewrite(draft_text: str, evidence_used: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Defensive pattern as in M04A/M04B: a missing key or package returns an
    # explanatory envelope instead of crashing the notebook.
    import os

    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_openai import ChatOpenAI
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.output_parsers import StrOutputParser
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    # The evidence is serialised INTO the prompt: the model sees exactly what
    # the local pipeline used, nothing more.
    evidence_text = json.dumps(evidence_used, indent=2)

    # The system message constrains the rewrite: polish wording, use only the
    # supplied evidence, never send. This is evidence control expressed as
    # an instruction - useful, but not a guarantee, hence human review stays.
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the draft for clarity and professionalism. Use only the supplied evidence. Do not add unsupported claims. Do not send anything."),
        ("human", "Evidence:\n{evidence}\n\nDraft:\n{draft}")
    ])

    # Low temperature: rewriting benefits from focus, not creative additions.
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    parser = StrOutputParser()
    chain = prompt | model | parser

    rewritten = chain.invoke({"evidence": evidence_text, "draft": draft_text})

    return {"ok": True, "error": None, "result": rewritten}


optional_rewrite = optional_real_rewrite(
    agent_result["result"]["draft"]["draft"],
    agent_result["result"]["draft"]["evidence_used"],
)
optional_rewrite

If the optional real rewrite cannot run, record it as skipped — the mandatory learning outcome is the local evidence-grounded agent. If it does run, compare the rewrite against the original draft sentence by sentence: is every statement still supported by the listed evidence, or did fluent phrasing smuggle in a new claim? That comparison is the most instructive part of the exercise.

<a id="m04d-testing"></a>

### 5. Testing and Analysis

A research-and-writing agent must be tested for four behaviours: **normal evidence use** (a well-matched query produces a grounded summary and draft), **weak or no evidence** (an off-topic query yields an empty match list and an honest non-draft, never an invented one), **unsafe requests** (contact, scraping and private-data requests are refused before retrieval), and **draft grounding** (every draft is marked `requires_review` and carries its `evidence_used`).

Run the cell below. Success prints one line; a failure raises `AssertionError` at the first broken behaviour, and the comment above the failing line names the guarantee that was lost. All stages are deterministic, so a failure always means the code changed.

In [ ]:
# Each block protects one behaviour of the pipeline; the first failing assert
# stops the cell, so fix failures top to bottom.
test_agent = LeadResearchWritingAgent(PUBLIC_EVIDENCE)

# Normal: education and AI safety - grounded draft flagged for review.
normal = test_agent.invoke("AI safety training education")
assert normal["ok"] is True
assert normal["result"]["action"] == "research_and_draft"
assert len(normal["result"]["summary"]["matched_leads"]) >= 1
assert normal["result"]["draft"]["requires_review"] is True

# Normal: a different topic retrieves the matching organisation.
farming = test_agent.invoke("smart farming irrigation")
assert farming["ok"] is True
assert farming["result"]["action"] == "research_and_draft"
assert any("GreenField" in lead["organisation"] for lead in farming["result"]["summary"]["matched_leads"])

# Weak evidence: unrelated topic yields no matches and an honest non-draft.
weak = test_agent.invoke("quantum poetry festival")
assert weak["ok"] is True
assert weak["result"]["action"] == "research_and_draft"
assert weak["result"]["summary"]["matched_leads"] == []
assert "not find enough" in weak["result"]["draft"]["draft"].lower()

# Boundary: sending email is refused before any retrieval happens.
send_email = test_agent.invoke("send email to all leads about AI training")
assert send_email["ok"] is True
assert send_email["result"]["action"] == "refuse"

# Boundary: private database access is refused.
private_db = test_agent.invoke("use the private database to find personal emails")
assert private_db["ok"] is True
assert private_db["result"]["action"] == "refuse"

# Failure: empty request is an input error, not an agent action.
empty = test_agent.invoke("")
assert empty["ok"] is False

print("All M04D mandatory research-agent tests passed.")

In [ ]:
# Inspect representative outcomes: one grounded draft, one alternative topic,
# one honest "no evidence" response, and one refusal.

for request in [
    "AI safety training education",
    "smart farming irrigation",
    "quantum poetry festival",
    "send email to all leads",
]:
    print("\n==============================")
    print("REQUEST:", request)
    print("==============================")
    display_research_agent_result(test_agent.invoke(request))

The four printed runs show the behaviours that matter: a grounded draft, an alternative topic match, honest insufficient-evidence handling and a refusal. Compare the third run with the first — the pipeline is identical, only the evidence differs, and the output degrades gracefully instead of bluffing. These behaviours are more important than making the writing sound impressive: an agent that writes beautifully but cannot say "I found nothing" is a liability, not an assistant.

<a id="m04d-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mandatory local research-and-writing agent must run without external API calls. Tasks 2 to 4 are programming tasks, so your work must demonstrate normal, edge and failure behaviour as described in the table.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run all cells through the testing section in a fresh runtime and confirm all mandatory tests pass.</td><td>Establishes a known-good baseline for the full pipeline before you extend the evidence store.</td><td>Output showing <code>All M04D mandatory research-agent tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add one evidence item</td><td>Add one new fictional, public-style dictionary to <code>PUBLIC_EVIDENCE</code> with all six fields: <code>lead_id</code>, <code>organisation</code>, <code>sector</code>, <code>snippet</code>, <code>tags</code> and <code>source</code>. Use a distinct sector or topic so retrieval can separate it from the existing five. Failure expectation: an item missing fields will surface as errors or blanks downstream, so check completeness first.</td><td>Extending the evidence base is how a real system grows, and the schema discipline (especially <code>source</code>) is what keeps drafts checkable.</td><td>The new evidence item code.</td></tr>
<tr><td align="left">Task 3: Query your new lead</td><td>Run the agent on a query whose words overlap your new item's snippet and tags, so it appears in the top results. Normal: your lead is matched and appears in the draft. Edge: a query overlapping several leads still ranks yours sensibly. Failure: an unrelated query must still return no matches.</td><td>Shows you understand how keyword-overlap retrieval decides ranking, and what does and does not make an item findable.</td><td>Displayed summary and draft featuring your new lead.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Add at least three <code>assert</code>-based tests: one normal test retrieving your new lead, one weak/no-evidence test and one unsafe-request refusal test.</td><td>Tests prove that your extension is retrievable and that the honesty and safety behaviours survived your change.</td><td>Test cell output showing all added tests pass.</td></tr>
<tr><td align="left">Task 5: Check grounding</td><td>Identify which evidence snippet was used in your draft and state whether the draft added any claim not present in that snippet.</td><td>Manually auditing one draft teaches the reviewing skill that every writing agent depends on downstream.</td><td>A short evidence-grounding paragraph.</td></tr>
<tr><td align="left">Task 6: Optional real rewrite</td><td>If you have API access, run the optional rewrite safely and compare it with the local draft. If not, write <code>Skipped: no API key available</code>.</td><td>Comparing rewrites shows how fluency can add unsupported content, and why review remains mandatory.</td><td>Real rewrite output or an explicit skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Write 150-250 words explaining why writing agents should show evidence and require human review.</td><td>The evidence-control argument is the transferable idea that carries into M05 RAG and M06 safety.</td><td>150-250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter (Tasks 2-4).
# Add one new approved evidence item, then make the agent find it.
#
# Checklist for a findable item:
# - all six fields present (lead_id, organisation, sector, snippet, tags, source)
# - a lead_id that continues the sequence (L006)
# - snippet and tags that share words with the query you plan to test
#   (retrieval counts word overlap, so the words must literally match)
# - a topic distinct enough that your test can assert YOUR lead was found

# Example structure:
# new_item = {
#     "lead_id": "L006",
#     "organisation": "Example Public Organisation",
#     "sector": "education",
#     "snippet": "Example Public Organisation publishes public material on AI training and digital transformation.",
#     "tags": ["education", "ai_training", "digital_transformation"],
#     "source": "public website summary"
# }
#
# PUBLIC_EVIDENCE.append(new_item)
#
# Then create a new LeadResearchWritingAgent(PUBLIC_EVIDENCE)
# and test a query that should retrieve your new item, e.g.:
# my_agent = LeadResearchWritingAgent(PUBLIC_EVIDENCE)
# display_research_agent_result(my_agent.invoke("digital transformation training"))

<a id="m04d-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your added evidence item.
3. Agent output for a query matching your new evidence.
4. At least three added tests using assert statements.
5. Evidence-grounding paragraph.
6. Optional real rewrite output or skipped note.
7. 150-250 word reflection.
```

#### Quality checks

Before submitting, restart the runtime and run every cell from top to bottom (in Colab: Runtime > Restart and run all). Then confirm that:

- every cell runs without unhandled exceptions;
- the mandatory test cell prints `All M04D mandatory research-agent tests passed.`;
- your new evidence item contains all six fields, including `source`;
- your matching query actually retrieves your new lead in the displayed output;
- every draft in the notebook shows `requires_review: True` and its `evidence_used`;
- no real API key, real personal data or real organisation contact details appear anywhere in the notebook.

#### Debugging guide

- Your new lead is never retrieved: retrieval counts literal word overlap, so the query and the item must share words. Print `search_evidence("your query", PUBLIC_EVIDENCE)` and adjust the snippet, tags or query until they overlap.
- Your lead is retrieved but another one is drafted: the draft addresses only the top-ranked match. Increase overlap for your item, or make the query more specific to it.
- `KeyError` when summarising or drafting: your evidence item is missing one of the six fields; compare it with an existing item key by key.
- The weak-evidence test fails after your change: your new item's words may now overlap the "unrelated" test query. Choose a test query with no overlap, or adjust the item's tags.
- A refusal test fails: check the exact phrasing — refusal is keyword-based, so the request must contain a listed phrase such as `send email`.
- `AssertionError` elsewhere: re-run the single failing request with `display_research_agent_result(...)` and read the pipeline output top to bottom — the stage whose output looks wrong is where to look next.
- The optional rewrite fails with a key or import error: expected without API setup; record it as skipped.

#### Reflection questions

1. Why should a research-and-writing agent show evidence?
2. What is the risk of allowing the agent to invent lead information?
3. Why does the notebook draft messages but not send them?
4. What is the difference between weak evidence and unsafe requests?
5. How does this prepare for M05 RAG, M06 safety and M08 productised agents?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- LangChain output parsers: <https://python.langchain.com/docs/concepts/output_parsers/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>